# globalGrowth EDA — RAN Pipeline

Basic exploratory data analysis on the `accumulator_samples` DuckDB store.
Validates correctness and covers EDA sufficient for FFI fuzz testing.

**Kernel:** `ran-venv` (RAN Pipeline Python 3.13)
**Data source:** `../data/ran_accumulator.duckdb`
**API:** `scripts.ran_data_api` — no raw SQL in this notebook.

In [9]:
import sys
from pathlib import Path

# Ensure scripts/ is importable
contracts_dir = Path.cwd().parent
if str(contracts_dir) not in sys.path:
    sys.path.insert(0, str(contracts_dir))

import duckdb
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # headless backend for nbconvert
import matplotlib.pyplot as plt

from scripts.ran_data_api import dataset_len, get_row, get_min, get_max, get_all, get_range, Row
from scripts.ran_utils import POOL_REGISTRY

DB_PATH = str(contracts_dir / 'data' / 'ran_accumulator.duckdb')
POOL_ID = POOL_REGISTRY['usdc-weth'].pool_id
Q128 = 2**128

conn = duckdb.connect(DB_PATH, read_only=True)
print(f'DB: {DB_PATH}')
print(f'Pool: {POOL_ID[:20]}...{POOL_ID[-8:]}')
print(f'DuckDB: {duckdb.__version__}, Pandas: {pd.__version__}')

DB: /home/jmsbpp/apps/ThetaSwap/thetaSwap-core-dev/.worktree/ranFromAngstrom/contracts/data/ran_accumulator.duckdb
Pool: 0xe500210c7ea6bfd9f6...08e3a657
DuckDB: 1.5.1, Pandas: 3.0.2


## 1. Load and Summary Stats

In [10]:
n = dataset_len(conn, POOL_ID)
row_min = get_min(conn, POOL_ID)
row_max = get_max(conn, POOL_ID)
all_rows = get_all(conn, POOL_ID)

df = pd.DataFrame([
    {'block_number': r.block_number, 'block_timestamp': r.block_timestamp, 'global_growth': r.global_growth}
    for r in all_rows
])

# Parse growth to numeric
df['growth_raw'] = df['global_growth'].apply(lambda x: int(x, 16))
df['growth_q128'] = df['growth_raw'] / Q128

print(f'Dataset length: {n:,}')
print(f'Block range: {row_min.block_number:,} to {row_max.block_number:,}')
print(f'Timestamp range: {row_min.block_timestamp} to {row_max.block_timestamp}')
print(f'Min growth (Q128): {df.growth_q128.min():.12e}')
print(f'Max growth (Q128): {df.growth_q128.max():.12e}')
df.head()

Dataset length: 37,678
Block range: 22,972,937 to 24,856,787
Timestamp range: 1753165727 to 1775914007
Min growth (Q128): 1.644047071674e-11
Max growth (Q128): 2.713228974476e-06


,block_number,block_timestamp,global_growth,growth_raw,growth_q128
0,22972937,1753165727,0x00000000000000000000000000000000000000001213...,5594402288787007442262793754,0.0
1,22972987,1753166327,0x00000000000000000000000000000000000000001213...,5594402288787007442262793754,0.0
2,22973037,1753166927,0x00000000000000000000000000000000000000001213...,5594402288787007442262793754,0.0
3,22973087,1753167527,0x00000000000000000000000000000000000000001213...,5594402288787007442262793754,0.0
4,22973137,1753168127,0x00000000000000000000000000000000000000001213...,5594402288787007442262793754,0.0


## 2. Correctness Checks

In [11]:
df['growth_delta'] = df['growth_raw'].diff()
violations = df[df['growth_delta'] < 0]
print(f'Monotonicity violations: {len(violations)}')
if len(violations) > 0:
    print('First 10 violations:')
    print(violations[['block_number', 'growth_raw', 'growth_delta']].head(10))
else:
    print('PASS: globalGrowth is monotonically non-decreasing')

Monotonicity violations: 0
PASS: globalGrowth is monotonically non-decreasing


In [12]:
df['ts_delta'] = df['block_timestamp'].diff()
ts_inversions = df[df['ts_delta'] < 0]
print(f'Timestamp inversions: {len(ts_inversions)}')
if len(ts_inversions) > 0:
    print('First 10 inversions:')
    print(ts_inversions[['block_number', 'block_timestamp', 'ts_delta']].head(10))
else:
    print('PASS: block_timestamp increases with block_number')

Timestamp inversions: 0
PASS: block_timestamp increases with block_number


In [13]:
zero_mask = df['growth_raw'] == 0
zero_blocks = df[zero_mask]['block_number'].tolist()
print(f'Zero-value rows: {len(zero_blocks)}')
if zero_blocks:
    # Find contiguous ranges
    ranges = []
    start = zero_blocks[0]
    prev = start
    for b in zero_blocks[1:]:
        if b - prev > 50:  # stride gap
            ranges.append((start, prev))
            start = b
        prev = b
    ranges.append((start, prev))
    print(f'Zero-value block ranges ({len(ranges)}):')
    for s, e in ranges:
        print(f'  blocks {s:,} to {e:,}')

Zero-value rows: 0


## 3. Edge Case Discovery (Fuzz Test Vectors)

In [14]:
# Largest growth delta
df['abs_delta'] = df['growth_delta'].abs()
max_spike_idx = df['abs_delta'].idxmax()
if pd.notna(max_spike_idx):
    spike = df.iloc[int(max_spike_idx)]
    print(f'Largest growth delta at index {int(max_spike_idx)}:')
    print(f'  block={spike.block_number:,} delta={spike.growth_delta:.0f} ({spike.growth_delta/Q128:.6e} Q128)')

# First non-zero growth
first_nonzero = df[df['growth_raw'] > 0].iloc[0] if (df['growth_raw'] > 0).any() else None
if first_nonzero is not None:
    print(f'\nFirst non-zero growth:')
    print(f'  block={first_nonzero.block_number:,} growth={first_nonzero.growth_q128:.6e} Q128')

# Long flat regions (10+ consecutive zero-delta strides)
flat_mask = df['growth_delta'] == 0
flat_runs = []
run_start = None
run_len = 0
for i, is_flat in enumerate(flat_mask):
    if is_flat:
        if run_start is None:
            run_start = i
        run_len += 1
    else:
        if run_start is not None and run_len >= 10:
            flat_runs.append((run_start, run_start + run_len - 1))
        run_start = None
        run_len = 0
if run_start is not None and run_len >= 10:
    flat_runs.append((run_start, run_start + run_len - 1))

print(f'\nLong flat regions (10+ consecutive zero-delta): {len(flat_runs)}')
for s, e in flat_runs[:5]:
    print(f'  indices {s} to {e} (blocks {df.iloc[s].block_number:,} to {df.iloc[e].block_number:,})')

# Timestamp gaps > 600s (expected for stride-50 at 12s/block)
ts_gaps = df[df['ts_delta'] > 600]
print(f'\nTimestamp gaps > 600s: {len(ts_gaps)}')
if len(ts_gaps) > 0:
    print('First 10:')
    print(ts_gaps[['block_number', 'block_timestamp', 'ts_delta']].head(10))

# Min/max growth as Q128 float
print(f'\nMin globalGrowth as Q128: {df.growth_q128.min():.12e}')
print(f'Max globalGrowth as Q128: {df.growth_q128.max():.12e}')

Largest growth delta at index 27677:
  block=24,356,787 delta=17751642296758707118039839539200 (5.216739e-08 Q128)

First non-zero growth:
  block=22,972,937 growth=1.644047e-11 Q128

Long flat regions (10+ consecutive zero-delta): 26
  indices 1 to 10 (blocks 22,972,987 to 22,973,437)
  indices 15 to 49 (blocks 22,973,687 to 22,975,387)
  indices 66 to 75 (blocks 22,976,237 to 22,976,687)
  indices 97 to 108 (blocks 22,977,787 to 22,978,337)
  indices 121 to 184 (blocks 22,978,987 to 22,982,137)

Timestamp gaps > 600s: 9622
First 10:
    block_number  block_timestamp  ts_delta
8       22973337       1753170539     612.0
11      22973487       1753172351     612.0
13      22973587       1753173563     612.0
15      22973687       1753174775     612.0
16      22973737       1753175387     612.0
17      22973787       1753176011     624.0
18      22973837       1753176635     624.0
21      22973987       1753178447     612.0
22      22974037       1753179071     624.0
24      22974137   

## 4. Visualizations

In [15]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Cumulative growth
axes[0].plot(df['block_number'], df['growth_q128'], linewidth=0.5)
axes[0].set_ylabel('globalGrowth (Q128)')
axes[0].set_title('Cumulative globalGrowth — USDC/WETH Pool')
axes[0].ticklabel_format(axis='x', style='plain')

# Delta histogram
deltas = df['growth_delta'].dropna()
deltas_q128 = deltas / Q128
axes[1].hist(deltas_q128[deltas_q128 > 0], bins=100, edgecolor='none', alpha=0.7)
axes[1].set_xlabel('Growth Delta (Q128)')
axes[1].set_ylabel('Count')
axes[1].set_title('Growth Delta Distribution (non-zero only)')

# Timestamp gap distribution
ts_deltas = df['ts_delta'].dropna()
axes[2].hist(ts_deltas, bins=100, edgecolor='none', alpha=0.7)
axes[2].set_xlabel('Timestamp Gap (seconds)')
axes[2].set_ylabel('Count')
axes[2].set_title('Timestamp Gap Distribution')

plt.tight_layout()
plt.savefig('/tmp/ran_eda_plots.png', dpi=100)
plt.show()
print('Plots saved to /tmp/ran_eda_plots.png')

Plots saved to /tmp/ran_eda_plots.png


/tmp/ipykernel_199850/3256835759.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Test Vector Summary

In [16]:
# Print specific (idx, blockNumber, blockTimestamp, globalGrowth) tuples
# worth testing on-chain as golden test vectors for the differential fuzz test.

vectors = []

# First and last
vectors.append(('first', 0))
vectors.append(('last', n - 1))

# First non-zero growth index
if first_nonzero is not None:
    first_nz_idx = df[df['growth_raw'] > 0].index[0]
    vectors.append(('first_nonzero', int(first_nz_idx)))

# Max spike
if pd.notna(max_spike_idx):
    vectors.append(('max_spike', int(max_spike_idx)))

# Zero-value row (if any)
if len(zero_blocks) > 0:
    zero_idx = df[zero_mask].index[0]
    vectors.append(('zero_growth', int(zero_idx)))

# Midpoint
vectors.append(('midpoint', n // 2))

print('=== Golden Test Vectors ===')
print(f'{"label":<20s} {"idx":>6s} {"blockNumber":>12s} {"blockTimestamp":>15s} {"globalGrowth"}')
print('-' * 100)
for label, idx in vectors:
    row = get_row(conn, POOL_ID, idx)
    print(f'{label:<20s} {idx:>6d} {row.block_number:>12d} {row.block_timestamp:>15d} {row.global_growth}')

=== Golden Test Vectors ===
label                   idx  blockNumber  blockTimestamp globalGrowth
----------------------------------------------------------------------------------------------------
first                     0     22972937      1753165727 0x0000000000000000000000000000000000000000121394c3c3566e7d59699e1a
last                  37677     24856787      1775914007 0x0000000000000000000000000000000000002d853ace66b3418e043455d4efcf
first_nonzero             0     22972937      1753165727 0x0000000000000000000000000000000000000000121394c3c3566e7d59699e1a
max_spike             27677     24356787      1769885807 0x00000000000000000000000000000000000021ed1a9188d2aa829dd1b4e38a81
midpoint              18839     23914887      1764546791 0x00000000000000000000000000000000000018a75dc7662abac28ecd56b024c6


In [17]:
conn.close()
print('Done. Connection closed.')

Done. Connection closed.
